In [1]:
from rustac import DuckdbClient
import geopandas as gpd 
import pandas as pd
import duckdb

In [2]:
import json
import shapely as sp

In [3]:
with open("aoi.json") as f:
    aoi = json.load(f)
    f.close()

In [4]:
aoi_geom = sp.geometry.shape(aoi)

In [5]:
aoi_geom.wkt

'POLYGON ((124350 485619.4, 125923.4 484605.2, 126312.3 485462.8, 125902.5 485646, 124589.4 485707.5, 124350 485619.4))'

In [6]:
client = DuckdbClient()
table = client.search_to_arrow(
    "ahn5.parquet",
    intersects=aoi,
)

In [7]:
import pyarrow.compute as pc

assets = table.column("assets").to_pylist()

In [8]:
assets[0]["data"]["href"]

'https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_124000_485000.COPC.LAZ'

In [9]:
def extract_copc_url(asset_row):

    if asset_row is None:
        return None

    #STRING SUPPORT DOESN"T WORK    
    if isinstance(asset_row, str):
        asset_dict = json.loads(asset_row)

    elif isinstance(asset_row, dict):
        asset_dict = asset_row

    else:
        asset_dict = dict(asset_row)

    return asset_dict.get("data", {}).get("href")

In [10]:
matched_copc_files = [
    u for u in map(extract_copc_url, assets)
    if u is not None
]

In [11]:
matched_copc_files

['https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_124000_485000.COPC.LAZ',
 'https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_125000_484000.COPC.LAZ',
 'https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_125000_485000.COPC.LAZ',
 'https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_126000_484000.COPC.LAZ',
 'https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_126000_485000.COPC.LAZ']

In [12]:

def format_shapely_for_pdal(geometry):
    """
    Transforms a Shapely geometry into optimized strings 
    compatible with PDAL's spatial filtering parameters.
    """
    # 1. Generate the 'bounds' string for the initial COPC octree pruning
    minx, miny, maxx, maxy = geometry.bounds
    pdal_bounds = f"([{minx}, {maxx}], [{miny}, {maxy}])"
    
    # 2. Generate the 'polygon' WKT string for the exact vector boundary clip
    # PDAL expects standard Well-Known Text (WKT)
    pdal_polygon_wkt = geometry.wkt
    
    return {
        "bounds": pdal_bounds,
        "polygon_wkt": pdal_polygon_wkt
    }

In [13]:
pdal_geom = format_shapely_for_pdal(aoi_geom)

In [14]:
pdal_bounds = pdal_geom["bounds"]
pdal_wkt = pdal_geom["polygon_wkt"]

In [15]:
#pipeline = []

#for fp in matched_copc_files:
#    pipeline.append({
#        "type": "readers.copc",
#        "filename": fp,
#        "bounds": pdal_bounds,
#    })


In [16]:
pipeline = {
    "type":"readers.copc",
    "bounds": pdal_bounds,
}

In [17]:
pipeline_json = json.dumps(pipeline)

pipeline_json_sql = pipeline_json.replace("'", "''")

#query = f"""
#SELECT *
#FROM PDAL_Pipeline('{pipeline_json_sql}',
#'')
#"""


In [18]:
conn = duckdb.connect(database=":memory:")
conn.execute("INSTALL spatial; LOAD spatial;")
conn.execute("INSTALL pdal FROM community; LOAD pdal;")
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute("SET threads=1;")

In [ ]:
table = []

for fp in matched_copc_files:
    print(fp)
    query = f"""
    SELECT *
    FROM PDAL_Pipeline('{fp}','[{pipeline_json}]');
    """
    arrow_table = conn.execute(query).arrow()
    print(f"Loaded {len(arrow_table)} points directly into PyArrow table.")
    table.append(arrow_table)
    

https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_124000_485000.COPC.LAZ


In [60]:
#arrow_table = conn.execute(query, [pdal_wkt]).arrow()
arrow_table = conn.execute(query).arrow()
print(f"Loaded {len(arrow_table)} points directly into PyArrow table.")

InvalidInputException: Invalid Input Error: File not found: [{"type": "readers.copc", "filename": "https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_124000_485000.COPC.LAZ", "bounds": "([124350.0, 126312.3], [484605.2, 485707.5])"}, {"type": "readers.copc", "filename": "https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_125000_484000.COPC.LAZ", "bounds": "([124350.0, 126312.3], [484605.2, 485707.5])"}, {"type": "readers.copc", "filename": "https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_125000_485000.COPC.LAZ", "bounds": "([124350.0, 126312.3], [484605.2, 485707.5])"}, {"type": "readers.copc", "filename": "https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_126000_484000.COPC.LAZ", "bounds": "([124350.0, 126312.3], [484605.2, 485707.5])"}, {"type": "readers.copc", "filename": "https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/01_LAZ/AHN5_C_126000_485000.COPC.LAZ", "bounds": "([124350.0, 126312.3], [484605.2, 485707.5])"}]

In [43]:
#items = gpd.GeoDataFrame.from_arrow(table)
#items = items.set_crs("EPSG:7415", allow_override=True)
#items

In [44]:
#items["stac_extensions"]

In [45]:
#items["assets"]

In [21]:
conn = duckdb.connect(database=":memory:")
conn.execute("INSTALL spatial; LOAD spatial;")
conn.execute("INSTALL pdal FROM community; LOAD pdal;")
conn.execute("INSTALL httpfs; LOAD httpfs;")

In [38]:
# Construct the optimized query. 
# We explicitly pass the 'bounds' parameter to read_pdal. This enables the 
# PDAL reader to prune the COPC octree *before* streaming points into DuckDB memory.
query = """
    SELECT * 
    FROM PDAL_Read(?, bounds=?) 
    WHERE ST_Within(ST_Point(X, Y), ST_GeomFromText(?))
"""

In [42]:
# 1. Dynamically generate the query string using UNION ALL
# This maps each file path into a single distinct PDAL_Read statement
#sub_queries = []
#for file_path in matched_copc_files:
    # Use DuckDB's MAP syntax to properly assign the 'bounds' parameter option
#    sub_queries.append(f"""
#        SELECT * 
#        FROM PDAL_Read('{file_path}', options = MAP(['box'], ['{pdal_bounds}']))
#    """)

## Combine all individual file loops into a single SQL stream execution block
#unified_pdal_query = " UNION ALL ".join(sub_queries)

## 2. Wrap the combined streams inside your final exact AoI spatial filter
#final_query = f"""
#    SELECT * 
#    FROM ({unified_pdal_query})
#    WHERE ST_Within(ST_Point(X, Y), ST_GeomFromText(?))
#"""



In [50]:
sub_queries = []
for file_path in matched_copc_files:
    # CRITICAL FIX: Wrap the pipeline pipeline steps inside a JSON Array bracket [ ... ]
    # This signals to DuckDB that it is reading a string, not a local file path.
    pipeline_list = [
        {
            "type": "readers.las",
            "filename": file_path
        },
        {
            "type": "filters.crop",
            "bounds": pdal_bounds
        }
    ]
    
    # Escape single quotes so it passes safely through DuckDB's compiler
    pipeline_json_string = json.dumps(pipeline_list).replace("'", "''")
    
    # CRITICAL FIX: Pass ONLY the json string. Do not supply the empty schema parameter ''.
    sub_queries.append(f"""
        SELECT * 
        FROM PDAL_Pipeline('{pipeline_json_string}')
    """)

# Combine independent file pipeline streams into one SQL execution plan
unified_pdal_query = " UNION ALL ".join(sub_queries)

final_query = f"""
    SELECT * 
    FROM ({unified_pdal_query})
    WHERE ST_Within(ST_Point(X, Y), ST_GeomFromText(?))
"""

In [51]:
# 3. Execute and instantly materialize to your in-memory PyArrow table
# Note: Since the file names and bounds are now baked directly into the query string,
# we only pass pdal_wkt as a dynamic execution parameter.
arrow_table = conn.execute(final_query, [pdal_wkt]).arrow()

print(f"Loaded {len(arrow_table)} points directly into PyArrow table.")

BinderException: Binder Error: No function matches the given name and argument types 'PDAL_Pipeline(VARCHAR)'. You might need to add explicit type casts.
	Candidate functions:
	PDAL_Pipeline(VARCHAR, VARCHAR, options : MAP(VARCHAR, VARCHAR))


LINE 5:         FROM PDAL_Pipeline('[{"type": "readers.las", "filename": "https...
                     ^